In [1]:

import re
import time
import requests
import pandas as pd

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from Bio.Align import PairwiseAligner
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment


# ============================================================
# 1. CONFIGURATION
# ============================================================

UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"
UNIPROT_ENTRY_URL = "https://rest.uniprot.org/uniprotkb"
PDB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
PDB_DATA_URL = "https://data.rcsb.org/rest/v1/core"

OUTPUT_PREFIX = "PDB_Structure_Collection"
PAGE_SIZE = 1000
REQUEST_DELAY = 0.10

# Sequence verification thresholds.
# Tune these carefully if collecting very short protein fragments.
MIN_SEQUENCE_IDENTITY = 0.70
MIN_TARGET_COVERAGE = 0.35

session = requests.Session()
session.headers.update({
    "User-Agent": "PDBStructureCollector/2.0",
    "Accept": "application/json"
})

retry = Retry(
    total=4,
    backoff_factor=0.7,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"]
)

session.mount("https://", HTTPAdapter(max_retries=retry))


# ============================================================
# 2. GENERAL HELPERS
# ============================================================

def get_json(url, params=None, timeout=60):
    response = session.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()


def normalize_name(value):
    if not value:
        return ""
    value = str(value).lower()
    value = value.replace("β", "beta").replace("α", "alpha")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def unique_strings(values):
    output, seen = [], set()

    for value in values:
        if not isinstance(value, str):
            continue

        value = value.strip()
        key = normalize_name(value)

        if key and key not in seen:
            seen.add(key)
            output.append(value)

    return output


# ============================================================
# 3. RESOLVE PROTEIN THROUGH UNIPROT
# ============================================================

def extract_uniprot_names(entry):
    names = []
    description = entry.get("proteinDescription", {})

    recommended = description.get("recommendedName", {})
    full_name = recommended.get("fullName", {}).get("value")

    if full_name:
        names.append(full_name)

    for short_name in recommended.get("shortNames", []):
        if short_name.get("value"):
            names.append(short_name["value"])

    for alternative in description.get("alternativeNames", []):
        full_name = alternative.get("fullName", {}).get("value")

        if full_name:
            names.append(full_name)

        for short_name in alternative.get("shortNames", []):
            if short_name.get("value"):
                names.append(short_name["value"])

    for gene in entry.get("genes", []):
        for field in (
            "geneName", "synonyms", "orderedLocusNames",
            "orfNames"
        ):
            value = gene.get(field, [])

            if isinstance(value, dict):
                value = [value]

            for item in value:
                if item.get("value"):
                    names.append(item["value"])

    return unique_strings(names)


def resolve_uniprot(protein_name):
    params = {
        "query": protein_name,
        "format": "json",
        "size": 100
    }

    data = get_json(UNIPROT_SEARCH_URL, params=params)
    results = data.get("results", [])

    if not results:
        raise ValueError(
            f"No UniProt result found for {protein_name!r}."
        )

    q = normalize_name(protein_name)

    def rank(entry):
        genes = []

        for gene in entry.get("genes", []):
            genes.append(normalize_name(
                gene.get("geneName", {}).get("value", "")
            ))

            genes.extend(
                normalize_name(x.get("value", ""))
                for x in gene.get("synonyms", [])
            )

        recommended = normalize_name(
            entry.get("proteinDescription", {})
            .get("recommendedName", {})
            .get("fullName", {})
            .get("value", "")
        )

        if q in genes:
            score = 0
        elif q == recommended:
            score = 1
        elif q in recommended:
            score = 2
        else:
            score = 5

        # Prefer reviewed records when identity matches equally.
        if entry.get("entryType") == "UniProtKB reviewed":
            score -= 0.1

        return score

    results.sort(key=rank)

    if rank(results[0]) >= 4:
        raise ValueError(
            "Could not identify an exact gene/protein-name match. "
            "Please enter a clearer protein name or gene symbol."
        )

    accession = results[0]["primaryAccession"]

    entry = get_json(
        f"{UNIPROT_ENTRY_URL}/{accession}.json"
    )

    sequence = (
        entry.get("sequence", {}).get("value", "")
        .upper()
    )

    aliases = unique_strings(
        [protein_name] + extract_uniprot_names(entry)
    )

    print("\nTARGET RESOLVED")
    print("-" * 60)
    print("Input name:", protein_name)
    print("UniProt accession:", accession)
    print("Recommended name:", (
        entry.get("proteinDescription", {})
        .get("recommendedName", {})
        .get("fullName", {})
        .get("value", "N/A")
    ))
    print("Sequence length:", len(sequence))
    print("Synonyms:", ", ".join(aliases))

    return accession, sequence, aliases


# ============================================================
# 4. RCSB SEARCH
# ============================================================

def search_pdb(query_node):
    """Retrieve all matching polymer-entity identifiers."""

    query = {
        "query": query_node,
        "return_type": "polymer_entity",
        "request_options": {
            "paginate": {"start": 0, "rows": PAGE_SIZE},
            "results_content_type": ["experimental"]
        }
    }

    identifiers = set()
    start = 0

    while True:
        query["request_options"]["paginate"]["start"] = start

        response = session.post(
            PDB_SEARCH_URL, json=query, timeout=90
        )
        response.raise_for_status()
        data = response.json()

        batch = [
            item["identifier"]
            for item in data.get("result_set", [])
            if item.get("identifier")
        ]

        identifiers.update(batch)
        total = data.get("total_count", 0)

        if not batch or start + len(batch) >= total:
            break

        start += len(batch)
        time.sleep(REQUEST_DELAY)

    return identifiers


def collect_candidate_entities(accession, aliases):
    candidates = set()

    # Primary search: direct UniProt accession cross-reference.
    accession_query = {
        "type": "terminal",
        "service": "text",
        "parameters": {
            "attribute":
                "rcsb_polymer_entity_container_identifiers."
                "uniprot_ids",
            "operator": "exact_match",
            "value": accession
        }
    }

    try:
        found = search_pdb(accession_query)
        candidates.update(found)
        print("Accession-based candidates:", len(found))
    except requests.RequestException as exc:
        print("Accession search warning:", exc)

    # Secondary search: exact phrases from UniProt names.
    for alias in aliases:
        if len(normalize_name(alias)) < 3:
            continue

        alias_query = {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute":
                    "rcsb_polymer_entity.pdbx_description",
                "operator": "contains_phrase",
                "value": alias
            }
        }

        try:
            found = search_pdb(alias_query)
            candidates.update(found)
            print(f"{alias}: {len(found)} search result(s)")
        except requests.RequestException as exc:
            print(f"Search warning for {alias!r}: {exc}")

        time.sleep(REQUEST_DELAY)

    print("Unique candidate entities:", len(candidates))
    return candidates


# ============================================================
# 5. FETCH PDB DATA
# ============================================================

def get_polymer_entity(pdb_id, entity_id):
    return get_json(
        f"{PDB_DATA_URL}/polymer_entity/{pdb_id}/{entity_id}"
    )


def get_pdb_entry(pdb_id):
    return get_json(f"{PDB_DATA_URL}/entry/{pdb_id}")


def get_entity_description(entity):
    return entity.get("rcsb_polymer_entity", {}).get(
        "pdbx_description", ""
    )


def get_entity_sequence(entity):
    polymer = entity.get("entity_poly", {})

    sequence = (
        polymer.get("pdbx_seq_one_letter_code_can")
        or polymer.get("pdbx_seq_one_letter_code")
        or ""
    )

    # Remove whitespace and non-sequence characters.
    return re.sub(r"[^A-Za-z]", "", sequence).upper()


def get_entity_uniprot_ids(entity):
    identifiers = (
        entity.get("rcsb_polymer_entity_container_identifiers", {})
        .get("uniprot_ids", [])
    ) or []

    return unique_strings(identifiers)


# ============================================================
# 6. SEQUENCE-BASED TARGET IDENTITY VERIFICATION
# ============================================================

aligner = PairwiseAligner()
aligner.mode = "local"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -5
aligner.extend_gap_score = -0.5


def sequence_identity_and_coverage(target_seq, pdb_seq):
    """
    Compare a PDB polymer sequence with the UniProt target.
    Returns identity among aligned residues and the fraction
    of the target sequence covered by the alignment.
    """

    if not target_seq or not pdb_seq:
        return 0.0, 0.0

    try:
        alignment = aligner.align(target_seq, pdb_seq)[0]
        indices = alignment.indices

        aligned_pairs = 0
        identical_pairs = 0
        target_positions = set()

        for i in range(indices.shape[1]):
            ti = indices[0, i]
            pi = indices[1, i]

            if ti < 0 or pi < 0:
                continue

            aligned_pairs += 1
            target_positions.add(int(ti))

            if target_seq[ti] == pdb_seq[pi]:
                identical_pairs += 1

        if aligned_pairs == 0:
            return 0.0, 0.0

        identity = identical_pairs / aligned_pairs
        coverage = len(target_positions) / len(target_seq)

        return identity, coverage

    except Exception:
        return 0.0, 0.0


def verify_target(entity, target_accession, target_seq, aliases):
    """
    Strict verification:
      1. Direct UniProt accession match; OR
      2. Strong sequence identity plus sufficient target coverage.

    A description mentioning the target or its inhibitor is
    NOT sufficient evidence.
    """

    uniprot_ids = get_entity_uniprot_ids(entity)

    if target_accession in uniprot_ids:
        return True, "UniProt accession match"

    pdb_seq = get_entity_sequence(entity)

    if pdb_seq and target_seq:
        identity, coverage = sequence_identity_and_coverage(
            target_seq, pdb_seq
        )

        if (
            identity >= MIN_SEQUENCE_IDENTITY
            and coverage >= MIN_TARGET_COVERAGE
        ):
            return True, (
                f"Sequence match: identity={identity:.1%}, "
                f"target coverage={coverage:.1%}"
            )

    # Never accept a candidate from name matching alone.
    return False, "Identity not verified"


# ============================================================
# 7. SOURCE ORGANISM ONLY
# ============================================================

def get_source_organism(entity):
    """
    Use only the polymer's source organism.
    Do NOT read rcsb_entity_host_organism, which describes
    the expression host.
    """

    organisms = []

    for item in entity.get("rcsb_entity_source_organism", []) or []:
        name = item.get("ncbi_scientific_name")

        if name:
            organisms.append(name)

    organisms = unique_strings(organisms)

    return "; ".join(organisms) if organisms else "N/A"


# ============================================================
# 8. EXPERIMENTAL METHOD
# ============================================================

def get_experimental_method(entry):
    methods = unique_strings([
        item.get("method", "")
        for item in (entry.get("exptl", []) or [])
        if item.get("method")
    ])

    return "; ".join(methods) if methods else "N/A"


def is_experimental_entry(entry):
    methods = get_experimental_method(entry).lower()

    # Keep X-ray, electron microscopy, NMR, and other
    # experimental methods; exclude theoretical-only models.
    if methods == "n/a":
        return False

    if "theoretical model" in methods:
        return False

    return True


# ============================================================
# 9. COLLECT VERIFIED PDB STRUCTURES
# ============================================================

def collect_pdb_structures(protein_name):
    accession, target_seq, aliases = resolve_uniprot(protein_name)

    candidates = collect_candidate_entities(accession, aliases)

    records = []
    verified_entries = {}
    checked_entries = set()

    for number, entity_identifier in enumerate(
        sorted(candidates), start=1
    ):
        try:
            pdb_id, entity_id = entity_identifier.split("_", 1)
            pdb_id = pdb_id.upper()

            entity = get_polymer_entity(pdb_id, entity_id)

            accepted, reason = verify_target(
                entity, accession, target_seq, aliases
            )

            if not accepted:
                description = get_entity_description(entity)
                print(
                    f"Rejected {pdb_id}_{entity_id}: "
                    f"{description} ({reason})"
                )
                continue

            if pdb_id not in checked_entries:
                entry = get_pdb_entry(pdb_id)

                if not is_experimental_entry(entry):
                    checked_entries.add(pdb_id)
                    continue

                checked_entries.add(pdb_id)

                verified_entries[pdb_id] = {
                    "entry": entry,
                    "organisms": [],
                    "uniprot_ids": [],
                    "verification": []
                }

            result = verified_entries[pdb_id]

            organism = get_source_organism(entity)
            if organism != "N/A":
                result["organisms"].extend(organism.split("; "))

            ids = get_entity_uniprot_ids(entity)
            result["uniprot_ids"].extend(ids)
            result["verification"].append(reason)

        except (requests.RequestException, ValueError, KeyError) as exc:
            print(f"Skipping {entity_identifier}: {exc}")

        if number % 100 == 0:
            print(f"Checked {number}/{len(candidates)} candidates")

        time.sleep(REQUEST_DELAY)

    for pdb_id, data in verified_entries.items():
        entry = data["entry"]

        accessions = unique_strings(data["uniprot_ids"])
        organisms = unique_strings(data["organisms"])

        records.append({
            "Protein Name": protein_name,
            "PDB ID": pdb_id,
            "PDB Link": f"https://www.rcsb.org/structure/{pdb_id}",
            "Source Organism": (
                "; ".join(organisms) if organisms else "N/A"
            ),
            "UniProt Accession": (
                "; ".join(accessions) if accessions else "N/A"
            ),
            "Method": get_experimental_method(entry)
        })

    columns = [
        "Protein Name",
        "PDB ID",
        "PDB Link",
        "Source Organism",
        "UniProt Accession",
        "Method"
    ]

    df = pd.DataFrame(records, columns=columns)

    if not df.empty:
        df = df.drop_duplicates(subset=["PDB ID"])
        df = df.sort_values("PDB ID").reset_index(drop=True)

    return df


# ============================================================
# 10. EXPORT EXCEL
# ============================================================

def save_to_excel(df, protein_name):
    safe_name = re.sub(
        r'[\\/*?:"<>|]+', "_", protein_name
    ).strip()

    output_file = f"{safe_name}_{OUTPUT_PREFIX}.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(
            writer, index=False, sheet_name="PDB_Structures"
        )

    wb = load_workbook(output_file)
    ws = wb["PDB_Structures"]

    header_fill = PatternFill(
        fill_type="solid", fgColor="1F4E78"
    )

    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = header_fill
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    link_column = 3

    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=link_column)

        if cell.value:
            cell.hyperlink = cell.value
            cell.style = "Hyperlink"

    widths = {
        "A": 24, "B": 12, "C": 48,
        "D": 32, "E": 28, "F": 30
    }

    for col, width in widths.items():
        ws.column_dimensions[col].width = width

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 32

    wb.save(output_file)

    print("\nCOLLECTION COMPLETE")
    print("Protein:", protein_name)
    print("Verified structures:", len(df))
    print("Output:", output_file)

    return output_file


# ============================================================
# 11. MAIN
# ============================================================

if __name__ == "__main__":
    protein_name = input(
        "Enter target protein name or gene symbol (e.g., CDK4): "
    ).strip()

    if not protein_name:
        raise ValueError("Protein name cannot be empty.")

    results_df = collect_pdb_structures(protein_name)

    if results_df.empty:
        print(
            "\nNo structures passed identity verification. "
            "Review the UniProt accession and search candidates; "
            "do not interpret this as proof that no structures exist."
        )
    else:
        save_to_excel(results_df, protein_name)
        print("\nPreview:")
        print(results_df.head(10).to_string(index=False))


Enter target protein name or gene symbol (e.g., CDK4):  PARP1



TARGET RESOLVED
------------------------------------------------------------
Input name: PARP1
UniProt accession: Q5RHR0
Recommended name: Poly [ADP-ribose] polymerase 1
Sequence length: 1013
Synonyms: PARP1, Poly [ADP-ribose] polymerase 1, PARP-1, ADP-ribosyltransferase diphtheria toxin-like 1, ARTD1, DNA ADP-ribosyltransferase PARP1, NAD(+) ADP-ribosyltransferase 1, ADPRT 1, Poly[ADP-ribose] synthase 1, Protein poly-ADP-ribosyltransferase PARP1
Accession search warning: 400 Client Error: Bad Request for url: https://search.rcsb.org/rcsbsearch/v2/query
PARP1: 4 search result(s)
Poly [ADP-ribose] polymerase 1: 116 search result(s)
Search warning for 'PARP-1': Expecting value: line 1 column 1 (char 0)
Search warning for 'ADP-ribosyltransferase diphtheria toxin-like 1': Expecting value: line 1 column 1 (char 0)
Search warning for 'ARTD1': Expecting value: line 1 column 1 (char 0)
Search warning for 'DNA ADP-ribosyltransferase PARP1': Expecting value: line 1 column 1 (char 0)
Search warn